<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; EDM&plus; Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">EDM&plus; NB04 &mdash; Legal Entities &amp; Relationships</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Builds 226 issuer legal entities, links instruments to their issuers, and assembles a 314-link corporate ownership hierarchy.</div>
</div>

<sub>EDM&plus; pack sequence: NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; <b>NB04</b> &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07</sub>

# NB04: Legal Entities & Relationships

**What this does:** Creates 226 issuer legal entities, links 69 instruments to their issuers, and builds a 314-link corporate hierarchy.

**Business context:** Legal entities represent the companies that issue securities. Relationships link instruments to their issuers and build the corporate ownership hierarchy (subsidiary → parent → ultimate parent). This enables concentration risk analysis ("how much exposure do we have to HSBC group across all subsidiaries?").

**Key patterns:**
- Legal entity identifiers need a property with `constraint_style='Identifier'`
- Instrument relationships must use `LusidInstrumentId` (resolved from ClientInternal)
- Hierarchy relationships use the `idTypeScope/idTypeCode/code` format

**Verify after running:** Instruments → search Apple → Relationships tab → select "Instrument Issuer" from dropdown.

In [ ]:
# Run this cell first — installs required packages (takes ~30 seconds, only needed once per session)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'lusid-sdk', 'lusid-workflow-sdk', 'finbourne-sdk-utilities', 'lusid-drive-sdk', 'lumipy'])
print("✅ Packages installed")

In [ ]:
# ============================================================================
# CREDENTIALS: edit secrets.json (NOT this notebook)
# ============================================================================
# Copy secrets.json.example to secrets.json and fill in your domain + PAT:
#   {
#     "api_url": "https://<YOUR_DOMAIN>.lusid.com/api",
#     "personal_access_token": "<YOUR_PAT>"
#   }
# To get a PAT: LUSID web app → profile icon (top right) → Personal Access Tokens → Create
# (Override the file location with the EDM_SECRETS_PATH environment variable.)

import json as _json, os as _os
_secrets_path = _os.environ.get("EDM_SECRETS_PATH", "secrets.json")
with open(_secrets_path) as _f:
    _secrets = _json.load(_f)
api_url = _secrets["api_url"]
personal_access_token = _secrets["personal_access_token"]

# ============================================================================
# Imports — do not edit below this line
# ============================================================================
import os, json
from datetime import datetime, timedelta, date, timezone
import datetime as dt
import pandas as pd
import pytz

import lusid as lu
import lusid.models as lm

try:
    import lusid_workflow as lwf
    import lusid_workflow.models as wf_models
except ImportError:
    lwf = None
    wf_models = None
    print("⚠️ lusid_workflow not available")

import lumipy as lp

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.4f}".format

# ============================================================================
# Authentication
# ============================================================================
if "<YOUR_DOMAIN>" in api_url or "<YOUR_PAT>" in personal_access_token or not personal_access_token:
    raise ValueError(
        "\n\n⛔ You need to edit the two lines at the top of this cell:\n"
        "   1. Replace <YOUR_DOMAIN> with your LUSID domain name\n"
        "   2. Replace <YOUR_PAT> with your Personal Access Token\n\n"
        "   To get a PAT: LUSID web app → profile icon → Personal Access Tokens → Create")

def get_factory(app='lusid'):
    module = __import__(app)
    # Each service has a different URL path
    url_map = {'lusid': '/api', 'lusid_workflow': '/workflow', 'lusid_drive': '/drive'}
    service_url = api_url.replace('/api', url_map.get(app, '/api'))
    config_loaders = [module.extensions.ArgsConfigurationLoader(
        api_url=service_url, access_token=personal_access_token)]
    return module.extensions.SyncApiClientFactory(config_loaders=config_loaders)

def get_lumi():
    luminesce_url = api_url.replace('/api', '/honeycomb')
    return lp.get_client(access_token=personal_access_token, api_url=luminesce_url)

# Initialise
lusid_factory = get_factory('lusid')

# Verify connection
try:
    api_instance = lusid_factory.build(lu.ApplicationMetadataApi)
    api_response = api_instance.get_lusid_versions()
    domain = api_response.links[0].href
    env_url = domain[0:domain.find('/app/')]
    print(f"✅ Connected to {env_url}")
    print(f"   API Version: {api_response.build_version}")
except Exception as e:
    print(f"⚠️ Connected but could not verify: {e}")

lumi = get_lumi()
print("✅ Luminesce ready")

---
## Build APIs, Create Identifier Property, Relationship Definitions

In [ ]:
legal_entities_api = lusid_factory.build(lu.LegalEntitiesApi)
property_definitions_api = lusid_factory.build(lu.PropertyDefinitionsApi)
rel_defs_api = lusid_factory.build(lu.RelationshipDefinitionsApi)
relationships_api = lusid_factory.build(lu.RelationshipsApi)
instruments_api = lusid_factory.build(lu.InstrumentsApi)

SCOPE = 'edm-training2'
DATA_DIR = 'data/'

# Create IssuerCode as an Identifier property
print("--- Ensuring IssuerCode property exists ---")
try:
    property_definitions_api.create_property_definition(
        create_property_definition_request=lm.CreatePropertyDefinitionRequest(
            domain='LegalEntity', scope=SCOPE, code='IssuerCode',
            display_name='Issuer Code',
            data_type_id=lm.ResourceId(scope='system', code='string'),
            description='Unique issuer identifier code',
            constraint_style='Identifier', life_time='Perpetual'))
    print("  ✅ Created: LegalEntity/edm-training2/IssuerCode")
except lu.ApiException as e:
    if 'AlreadyExists' in str(e.body):
        print("  ℹ️  Already exists")
    else:
        error = json.loads(e.body)
        print(f"  ❌ Error: {error.get('title', e.reason)}")

# Relationship Definitions
print("\n--- Creating Relationship Definitions ---")
for rd in [
    {'code': 'Issuer', 'source': 'Instrument', 'target': 'LegalEntity', 'outward': 'is issued by', 'inward': 'issues'},
    {'code': 'ImmediateParent', 'source': 'LegalEntity', 'target': 'LegalEntity', 'outward': 'is subsidiary of', 'inward': 'is immediate parent of'},
    {'code': 'UltimateParent', 'source': 'LegalEntity', 'target': 'LegalEntity', 'outward': 'is ultimately owned by', 'inward': 'is ultimate parent of'},
]:
    try:
        rel_defs_api.create_relationship_definition(
            create_relationship_definition_request=lm.CreateRelationshipDefinitionRequest(
                scope=SCOPE, code=rd['code'], source_entity_type=rd['source'],
                target_entity_type=rd['target'], display_name=rd['code'],
                outward_description=rd['outward'], inward_description=rd['inward'],
                life_time='Perpetual', relationship_cardinality='ManyToMany'))
        print(f"  ✅ Created: {rd['code']}")
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body):
            print(f"  ℹ️  Already exists: {rd['code']}")
        else:
            body = e.body[:300] if e.body else str(e.reason)
            print(f"  ❌ Error {rd['code']}: {body}")

---
## Create Issuer Legal Entities

In [ ]:
print("--- Creating Issuer Legal Entities ---")
df_issuers = pd.read_csv(f'{DATA_DIR}issuers.csv')
print(f"  Read {len(df_issuers)} issuers")

created = 0
errors = 0
for idx, row in df_issuers.iterrows():
    code_val = str(row['IssuerCode']).strip()
    name = str(row.get('IssuerName', code_val)).strip()
    identifiers = {
        f'LegalEntity/{SCOPE}/IssuerCode': lm.ModelProperty(
            key=f'LegalEntity/{SCOPE}/IssuerCode',
            value=lm.PropertyValue(label_value=code_val))
    }
    props = {}
    lei = str(row.get('LEI', '')).strip()
    if lei and lei != 'nan':
        props[f'LegalEntity/{SCOPE}/LEI'] = lm.ModelProperty(
            key=f'LegalEntity/{SCOPE}/LEI', value=lm.PropertyValue(label_value=lei))
    country = str(row.get('IssuerCountry', '')).strip()
    if country and country != 'nan':
        props[f'LegalEntity/{SCOPE}/IssuerCountry'] = lm.ModelProperty(
            key=f'LegalEntity/{SCOPE}/IssuerCountry', value=lm.PropertyValue(label_value=country))
    try:
        legal_entities_api.upsert_legal_entity(
            upsert_legal_entity_request=lm.UpsertLegalEntityRequest(
                identifiers=identifiers, display_name=name,
                description=f'Issuer: {name}', properties=props if props else None))
        created += 1
    except lu.ApiException as e:
        errors += 1
        if errors <= 3:
            body = e.body[:300] if e.body else str(e.reason)
            print(f"  ❌ Error on {code_val}: {body}")
print(f"  Upserted {created} issuers ({errors} errors)")

---
## Create Instrument to Issuer Relationships

In [ ]:
print("--- Creating Instrument to Issuer Relationships ---")
df_i2i = pd.read_csv(f'{DATA_DIR}issuer-to-instrument.csv')
print(f"  Read {len(df_i2i)} mappings")

# Resolve ClientInternal -> LUID
ci_codes = [str(row['ClientInternal']).strip() for _, row in df_i2i.iterrows()]
ci_to_luid = {}
try:
    resp = instruments_api.get_instruments(identifier_type='ClientInternal', request_body=ci_codes, scope='default')
    for ci, inst in resp.values.items():
        ci_to_luid[ci] = inst.lusid_instrument_id
    print(f"  Resolved {len(ci_to_luid)} instruments to LUIDs")
except Exception as e:
    print(f"  Could not resolve: {e}")

rel_created = 0
rel_errors = 0
for idx, row in df_i2i.iterrows():
    issuer_code = str(row['IssuerCode']).strip()
    ci = str(row['ClientInternal']).strip()
    luid = ci_to_luid.get(ci)
    if not luid:
        rel_errors += 1
        continue
    try:
        relationships_api.create_relationship(
            scope=SCOPE, code='Issuer',
            create_relationship_request=lm.CreateRelationshipRequest(
                source_entity_id={'scope': 'default', 'identifierType': 'LusidInstrumentId', 'identifierValue': luid},
                target_entity_id={'idTypeScope': SCOPE, 'idTypeCode': 'IssuerCode', 'code': issuer_code}))
        rel_created += 1
    except lu.ApiException as e:
        if 'AlreadyExists' in str(e.body): rel_created += 1
        else: rel_errors += 1
print(f"  Created {rel_created} relationships ({rel_errors} errors)")

---
## Build Corporate Hierarchy

In [ ]:
print("--- Building Corporate Hierarchy ---")
hierarchy_created = 0
for idx, row in df_issuers.iterrows():
    code_val = str(row['IssuerCode']).strip()
    for rel_code, col in [('ImmediateParent', 'immediateParent'), ('UltimateParent', 'ultimateParent')]:
        parent = str(row.get(col, '')).strip()
        if parent and parent != code_val and parent != 'nan':
            try:
                relationships_api.create_relationship(
                    scope=SCOPE, code=rel_code,
                    create_relationship_request=lm.CreateRelationshipRequest(
                        source_entity_id={'idTypeScope': SCOPE, 'idTypeCode': 'IssuerCode', 'code': code_val},
                        target_entity_id={'idTypeScope': SCOPE, 'idTypeCode': 'IssuerCode', 'code': parent}))
                hierarchy_created += 1
            except: pass
print(f"  Created {hierarchy_created} hierarchy relationships")
print(f"\n✅ NB04 Complete")